<!-- <img src="https://s3-eu-west-1.amazonaws.com/dani.gon.med/cidaen/images/cidaenNB.png" alt="Logo CiDAEN" align="right"> -->

<h1><font color="#113D68" size=5>Módulo 14</font></h1>



<h1><font color="#113D68" size=6>Capstone: Despliegue de modelos en producción</font></h1>

<br><br>
<div style="text-align: right">
  <font color="#113D68" size=3>Daniel González</font><br>
  <font color="#113D68" size=3>Máster en Ciencia de Datos e Ingienería de Datos en la Nube</font><br>
  <font color="#113D68" size=3>Universidad de Castilla-La Mancha</font>
</div>

En este proyecto vamos a poner en práctica lo visto sobre MLOps, donde partiremos de unos datos en crudo y tendremos que usarlos para entrenar un modelo de Machine Learning y ponerlo en producción para que lo usarlo en una aplicación.

El proyecto consisten en lo siguientes pasos:

1. **Datos**: carga y limpieza de los datos.
2. **MLOps**: entrenamiento de un modelo usando *mlflow* registrando los parámetros utilizados, las métricas obtenidas y el modelo resultante.
3. **Despliegue**: desplegaremos el modelo en una aplicación, para ello podremos desplegarlo embebido en la aplicación o bajo una API.

Cada uno de los pasos está definido y explicado, donde se guía con todo lo que hay que hacer en cada momento.

In [1]:
import pandas as pd
import mlflow
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score

import tensorflow as tf

## 1. Datos

Los datos que utilizaremos son datos relacionados con enfermedades cardiovaculares, la primera causa de muerte a nivel mundial, se estima que 17,9 millones de personas fallecen cada año por este tipo de enfermedades, lo que representa el 31% de las muertes en todo el mundo. Las personas con enfermedad cardiovascular o que se encuentran en alto riesgo cardiovascular (debido a la presencia de uno o más factores de riesgo como hipertensión, diabetes, hiperlipidemia o enfermedad ya establecida) necesitan una detección preventia, por ellos crearemos un modelo de aprendizaje automático que pueda ser de gran ayuda.

Contamos con un dataset con datos realacionados con estas enfermedades con lo que intentaremos para predecir la probabilidad de padecer enfermedades del corazón. Contamos con 12 variables en total, pero utilizaremos 7 de ellas, las cuales son:

- *Age*: edad de la persona.
- *Sex*: sexo de la persona *[M: Male, F: Female]*.
- *RestingBP*: presión de la sangre en reposo.
- *Cholesterol*: colesterol.
- *FastingBS*:  nivel de azucar en ayunas *[1: if FastingBS > 120 mg/dl, 0: otherwise]*
- *MaxHR*:  frecuencia cardíaca máxima alcanzada.
- *HeartDisease*: variable objetivo [1: heart disease, 0: Normal]



### 1.1 Carga los datos con pandas y filtra las columnas que vamos a usar

In [2]:
ruta = "data/heart.csv"
columnas_utilizadas = ["Age", "Sex", "RestingBP", "Cholesterol", "FastingBS", "MaxHR", "HeartDisease"]
df = pd.read_csv(ruta, usecols=columnas_utilizadas)
df.head()

,Age,Sex,RestingBP,Cholesterol,FastingBS,MaxHR,HeartDisease
0,40,M,140,289,0,172,0
1,49,F,160,180,0,156,1
2,37,M,130,283,0,98,0
3,48,F,138,214,0,108,1
4,54,M,150,195,0,122,0


### 1.2 Transforma la columna *Sex* para que sea numércia *[M: 0, F: 1]*

In [3]:
df["Sex"] = df["Sex"].map({"M": 0, "F": 1})


### 1.3 Separa las columnas predictoras y la columna a predecir (objetivo)

In [4]:
X = df.drop("HeartDisease", axis=1)
y = df["HeartDisease"]


### 1.4 Separa los datos en un conjunto de train y en un conjunto de test (25% de los datos)

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

## 2. MLOps

En este paso vamos a utilizar la herramienta **_mlflow_** para utilizar una metodología de MLOps donde registremos los entenamientos realizados y los modelos creados, de esta forma tendremos una trazabilidad completa de los experimentos que realizamos.

Para realizar esta sección se puede completar usando uno de estos 2 algotimos:
- *RandomForestClassifier* usando *sklearn*
- *Red neuronal con capas densas (Dense)* usando *tensorflow.keras*

Cuando elijamos qué algoritmo utilizar tendemos que realizar lo siguiente:
1. Crear un experimento de *mlflow*.
2. Lanzar un experimento donde:
    - Registremos los parámetros que se indican en *mlflow*.
    - Entrenemos el modelo.
    - Registremos las métricas que se indican en *mlflow*.
    - Registremos el modelo creado en *mlflow*.
3. Después de lanzar un entrenamiento, intenta modificar los parámetros para mejorar los resultados (no es necesario) y lanza otro entrenamiento.

Recuerda visualizar como se ha registrado cada entrenamiento con la interfaz de *mlflow*: `mlflow server`.

A continuación puedes ver un diagrama de la tarea a realizar:

![](images/mlflow-training.png)

### 2.0 Establecer servidor de mlflow

Establece la url del servidor donde se está ejecutando mlflow:

In [6]:
mlflow.set_tracking_uri("http://localhost:5000")

### 2.1 Entrenamiento del modelo y registro del experimento

#### [Opción 1] Random Forest con scikit-learn

Entrena un modelo *RandomForestClassifier* de *sklearn* con los siguientes **parámetros y registralos en _mlflow_**:

- `n_estimators = 124`
- `min_samples_split = 10`
- `min_samples_leaf = 2`
- `max_features = 'sqrt'`
- `max_depth = 90`
- `bootstrap = False`

**Registra las siguientes métricas en _mlflow_**:

- `train_accuracy`
- `train_precision`
- `train_recall`
- `test_accuracy`
- `test_precision`
- `test_recall`

In [21]:
mlflow.set_experiment("heart-disease-classifier")

params = {
    "n_estimators": 124,
    "min_samples_split": 10,
    "min_samples_leaf": 2,
    "max_features": "sqrt",
    "max_depth": 90,
    "bootstrap": False
}

with mlflow.start_run():
    model = RandomForestClassifier(**params, random_state=42)
    model.fit(X_train, y_train)


    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    mlflow.log_param("model_type", "RandomForest")
    for key, value in params.items():
        mlflow.log_param(key, value)

    mlflow.log_metric("train_accuracy", accuracy_score(y_train, y_train_pred))
    mlflow.log_metric("train_precision", precision_score(y_train, y_train_pred))
    mlflow.log_metric("train_recall", recall_score(y_train, y_train_pred))

    mlflow.log_metric("test_accuracy", accuracy_score(y_test, y_test_pred))
    mlflow.log_metric("test_precision", precision_score(y_test, y_test_pred))
    mlflow.log_metric("test_recall", recall_score(y_test, y_test_pred))

    mlflow.sklearn.log_model(model, "model",registered_model_name="model_random_forest")




2025/06/28 16:35:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/06/28 16:35:47 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Successfully registered model 'model_random_forest'.
2025/06/28 16:35:48 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: model_random_forest, version 1
Created version '1' of model 'model_random_forest'.


🏃 View run fortunate-sheep-10 at: http://localhost:5000/#/experiments/461735160528471256/runs/3967f3aeaa3b4907a4962d1f5347d7d5
🧪 View experiment at: http://localhost:5000/#/experiments/461735160528471256


#### [Opción 2] DNN con Tensorflow

Entrena un modelo *RandomForestClassifier* de *sklearn* con los siguientes **parámetros y registralos en _mlflow_**:

- `n_layers = 4`
- `nodes = [32, 32, 16, 8]`
- `optimizer = 'adam'`
- `loss = 'binary_crossentropy'`

**Registra las siguientes métricas en _mlflow_**:

- `train_accuracy`
- `train_precision`
- `train_recall`
- `test_accuracy`
- `test_precision`
- `test_recall`

In [23]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam

n_layers = 4
nodes = [32, 32, 16, 8]
optimizer = 'adam'
loss = 'binary_crossentropy'
epochs = 50
batch_size = 16

with mlflow.start_run():
    # Construir modelo secuencial
    model = Sequential()
    model.add(Dense(nodes[0], activation='relu', input_shape=(X_train.shape[1],)))
    for n in nodes[1:]:
        model.add(Dense(n, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))  
    model.compile(optimizer=optimizer, loss=loss, metrics=['accuracy'])

 
    model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, verbose=0)


    y_train_pred = (model.predict(X_train) > 0.5).astype("int32")
    y_test_pred = (model.predict(X_test) > 0.5).astype("int32")

   
    mlflow.log_param("model_type", "DenseNN")
    mlflow.log_param("n_layers", n_layers)
    mlflow.log_param("nodes", nodes)
    mlflow.log_param("optimizer", optimizer)
    mlflow.log_param("loss", loss)
    mlflow.log_param("epochs", epochs)
    mlflow.log_param("batch_size", batch_size)

    mlflow.log_metric("train_accuracy", accuracy_score(y_train, y_train_pred))
    mlflow.log_metric("train_precision", precision_score(y_train, y_train_pred))
    mlflow.log_metric("train_recall", recall_score(y_train, y_train_pred))

    mlflow.log_metric("test_accuracy", accuracy_score(y_test, y_test_pred))
    mlflow.log_metric("test_precision", precision_score(y_test, y_test_pred))
    mlflow.log_metric("test_recall", recall_score(y_test, y_test_pred))

   
    mlflow.tensorflow.log_model(model, "model_tensorflow",registered_model_name="model_tensorflow")

c:\Users\usuario\anaconda3\envs\.caps81\Lib\site-packages\keras\src\layers\core\dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 


2025/06/28 16:36:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/06/28 16:36:39 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.
2025/06/28 16:36:54 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Successfully registered model 'model_tensorflow'.
2025/06/28 16:36:55 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: model_tensorflow, version 1
Created version '1' of model 'model_tensorflow'.


🏃 View run languid-steed-137 at: http://localhost:5000/#/experiments/461735160528471256/runs/24bb42c1b2244a308ceacb858b4a5fdf
🧪 View experiment at: http://localhost:5000/#/experiments/461735160528471256


### 2.2 Registrar el modelo en producción

Registra el modelo con el *alias* `prod` para utilizarlo posteriormente en la aplicación.

**Nota**: es recomendable probar ha realizar predicciones con el modelo en `prod`.

In [24]:
from mlflow.tracking import MlflowClient

client = MlflowClient()
client.set_registered_model_alias("model_random_forest", "prod", "1")


In [28]:
import mlflow.pyfunc
model = mlflow.pyfunc.load_model("models:/model_random_forest@prod")
data = pd.DataFrame([{
    "Age": 50,
    "Sex": 0,              # 0: Male, 1: Female
    "RestingBP": 130,
    "Cholesterol": 250,
    "FastingBS": 1,        # 1: azúcar en ayunas > 120
    "MaxHR": 150
}])
prediction = model.predict(data)
print(f"Predicción: {prediction[0]}")


c:\Users\usuario\anaconda3\envs\.caps81\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Predicción: 0


## 3. Despliegue

Ahora vamos a desplegar el modelo que hemos creado. El modelo lo desplegaremos en una aplicación de *Streamlit*. La aplicación tiene que tener esta pinta:

![](images/app.png)

La aplicación usará internamente el modelo que hemos creado en el paso anterior, el cual lo usará para predecir si una persona tiene probabilidades de padecer problemas del cozarón o no. El modelo podemos desplegarlo usando uno de estos 2 patrones:

![](images/deployment.png)